<a href="https://colab.research.google.com/github/Pazidu/Research-Project/blob/main/newType_fixed2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/drive')

Mounted at /drive


In [2]:
import os
import shutil
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras import mixed_precision
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input

tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.test.gpu_device_name())

TensorFlow version: 2.20.0
GPU: /device:GPU:0


In [3]:
BASE            = "/content/newdata"
IMG_SRC         = "/drive/MyDrive/Colab Notebooks/newdata"
CHECKPOINT_DIR  = "/drive/MyDrive/checkpoints"
MODEL_SAVE_PATH = "/drive/MyDrive/Colab Notebooks/Models/dermoscopy/efficientnetv2s_dual_branch_attn.keras"

In [4]:
if os.path.exists(BASE):
    shutil.rmtree(BASE)
shutil.copytree(IMG_SRC, BASE)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [5]:
mixed_precision.set_global_policy("float32")

BATCH_SIZE       = 16
IMAGE_SIZE       = 256
FUSION_LAYER     = "block4c_add"
EPOCHS           = 30

N_TRAIN_TOTAL    = 7122 + 890
STEPS_PER_EPOCH  = N_TRAIN_TOTAL // BATCH_SIZE

CLASS_ORDER      = ["melanoma", "non_melanoma"]

In [6]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.1),
], name="data_augmentation")

In [7]:
def add_edge_map(image, label):
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    gray  = tf.image.rgb_to_grayscale(image)
    sobel = tf.image.sobel_edges(gray)
    edge  = tf.sqrt(tf.reduce_sum(tf.square(sobel), axis=-1))
    edge  = edge / (tf.reduce_max(edge) + 1e-6)
    return (image, edge), label

In [8]:
def make_class_stream(base_path, only_class):
    """Infinite stream containing only `only_class` images, correctly labelled."""
    ds = tf.keras.preprocessing.image_dataset_from_directory(
        base_path,
        image_size=(IMAGE_SIZE, IMAGE_SIZE),
        batch_size=BATCH_SIZE // 2,
        label_mode="categorical",
        shuffle=True,
        class_names=CLASS_ORDER,
    )
    ds = ds.filter(lambda x, y: tf.equal(tf.argmax(y[0]), only_class))
    ds = ds.repeat()
    ds = ds.map(
        lambda x, y: (data_augmentation(x, training=True), y),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    return ds


melanoma_stream     = make_class_stream(f"{BASE}/train", only_class=0)
non_melanoma_stream = make_class_stream(f"{BASE}/train", only_class=1)

balanced_ds = tf.data.Dataset.sample_from_datasets(
    [melanoma_stream, non_melanoma_stream],
    weights=[0.5, 0.5],
)
balanced_ds = balanced_ds.map(add_edge_map, num_parallel_calls=tf.data.AUTOTUNE)
balanced_ds = balanced_ds.prefetch(tf.data.AUTOTUNE)


Found 8012 files belonging to 2 classes.
Found 8012 files belonging to 2 classes.


In [9]:
def prepare_eval_dataset(path):
    ds = tf.keras.preprocessing.image_dataset_from_directory(
        path,
        image_size=(IMAGE_SIZE, IMAGE_SIZE),
        batch_size=BATCH_SIZE,
        label_mode="categorical",
        shuffle=False,
        class_names=CLASS_ORDER,
    )
    ds = ds.map(add_edge_map, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


val_ds  = prepare_eval_dataset(f"{BASE}/valid")
test_ds = prepare_eval_dataset(f"{BASE}/test")

Found 1001 files belonging to 2 classes.
Found 1002 files belonging to 2 classes.


In [10]:
def channel_attention_module(x, reduction_ratio=24, name="channel_attention"):
    """
    Squeeze-and-excitation style channel attention, as used in the paper.
    GAP -> FC(C/r) -> FC(C) -> Sigmoid -> multiply back into input feature map.
    """
    channels = x.shape[-1]
    gap = layers.GlobalAveragePooling2D(name=f"{name}_gap")(x)
    gap = layers.Reshape((1, 1, channels), name=f"{name}_reshape")(gap)
    fc1 = layers.Conv2D(max(channels // reduction_ratio, 1), 1,
                         activation="relu", name=f"{name}_fc1")(gap)
    fc2 = layers.Conv2D(channels, 1, activation="sigmoid",
                         name=f"{name}_fc2")(fc1)
    return layers.Multiply(name=f"{name}_scale")([x, fc2])


In [11]:
def create_dual_model(steps_per_epoch):
    # --- RGB branch ---
    rgb_input  = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3), name="rgb_input")
    base_model = EfficientNetV2S(
        include_top=False,
        weights="imagenet",
        input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    )

    for layer in base_model.layers[:-40]:
        layer.trainable = False

    feature_extractor = tf.keras.Model(
        inputs=base_model.input,
        outputs=base_model.get_layer(FUSION_LAYER).output,
    )
    middle_feature = feature_extractor(rgb_input)

    # --- Edge branch ---
    edge_input = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 1), name="edge_input")
    x = layers.Conv2D(32,  3, activation="relu", padding="same")(edge_input)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(64,  3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
    x = layers.Resizing(middle_feature.shape[1], middle_feature.shape[2])(x)
    x = layers.Conv2D(middle_feature.shape[-1], 1, padding="same")(x)

    # --- Feature fusion + channel attention ───────────────────────────────────
    fused = layers.Concatenate()([middle_feature, x])
    fused = layers.Conv2D(256, 3, activation="relu", padding="same")(fused)
    fused = layers.BatchNormalization()(fused)
    fused = channel_attention_module(fused, reduction_ratio=24)   # <-- paper's attention module
    fused = layers.GlobalAveragePooling2D()(fused)
    fused = layers.Dense(
        256, activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(1e-4),
    )(fused)
    fused   = layers.Dropout(0.5)(fused)
    outputs = layers.Dense(2, activation="softmax")(fused)

    cosine_lr = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=2e-4,
        decay_steps=EPOCHS * steps_per_epoch,
    )

    model = tf.keras.Model(inputs=[rgb_input, edge_input], outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(cosine_lr),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.Recall(name="sensitivity"),                # TP/(TP+FN), melanoma = class 0
            tf.keras.metrics.Recall(name="specificity", class_id=1),    # TN/(TN+FP), non_melanoma = class 1
            tf.keras.metrics.Precision(name="precision"),
        ],
    )
    return model

In [12]:
model = create_dual_model(steps_per_epoch=STEPS_PER_EPOCH)
model.summary()

82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ edge_input          │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 256, 256,  │        320 │ edge_input[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 128, 128,  │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 128, 128,  │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 64, 64,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 64, 64,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rgb_input           │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resizing (Resizing) │ (None, 16, 16,    │          0 │ conv2d_2[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_1        │ (None, 16, 16,    │  1,317,880 │ rgb_input[0][0]   │
│ (Functional)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 16, 16,    │     16,512 │ resizing[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 16, 16,    │          0 │ functional_1[0][… │
│ (Concatenate)       │ 256)              │            │ conv2d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 16, 16,    │    590,080 │ concatenate[0][0] │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 16, 16,    │      1,024 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ channel_attention_… │ (None, 256)       │          0 │ batch_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ channel_attention_… │ (None, 1, 1, 256) │          0 │ channel_attentio… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ channel_attention_… │ (None, 1, 1, 10)  │      2,570 │ channel_attentio… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ channel_attention_… │ (None, 1, 1, 256) │      2,816 │ channel_attentio

 Total params: 2,089,860 (7.97 MB)

 Trainable params: 771,468 (2.94 MB)

 Non-trainable params: 1,318,392 (5.03 MB)

In [13]:
checkpoint_best = ModelCheckpoint(
    filepath=f"{CHECKPOINT_DIR}/best_dual_{FUSION_LAYER}_attn_run9.keras",
    monitor="val_auc",
    save_best_only=True,
    verbose=1,
)

early_stopping = EarlyStopping(
    monitor="val_auc",
    patience=10,
    restore_best_weights=True,
    verbose=1,
)

In [14]:
history = model.fit(
    balanced_ds,
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_data=val_ds,
    callbacks=[checkpoint_best, early_stopping],
)

Epoch 1/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 417ms/step - accuracy: 0.8088 - auc: 0.8871 - loss: 0.4758 - precision: 0.8088 - sensitivity: 0.8088 - specificity: 0.9211
Epoch 1: val_auc improved from None to 0.86152, saving model to /drive/MyDrive/checkpoints/best_dual_block4c_add_attn_run9.keras

Epoch 1: finished saving model to /drive/MyDrive/checkpoints/best_dual_block4c_add_attn_run9.keras
500/500 ━━━━━━━━━━━━━━━━━━━━ 285s 509ms/step - accuracy: 0.8486 - auc: 0.9238 - loss: 0.4264 - precision: 0.8486 - sensitivity: 0.8486 - specificity: 0.9563 - val_accuracy: 0.7463 - val_auc: 0.8615 - val_loss: 0.5211 - val_precision: 0.7463 - val_sensitivity: 0.7463 - val_specificity: 0.7438
Epoch 2/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 371ms/step - accuracy: 0.8573 - auc: 0.9336 - loss: 0.4033 - precision: 0.8573 - sensitivity: 0.8573 - specificity: 0.9584
Epoch 2: val_auc improved from 0.86152 to 0.87738, saving model to /drive/MyDrive/checkpoints/best_dual_block4c_add_attn_run9.keras

Epoch 2: fini

In [15]:
print("\n── Test results (run 8 pipeline + channel attention) ──")
results = model.evaluate(test_ds)
for name, val in zip(model.metrics_names, results):
    print(f"  {name}: {val:.4f}")

model.save(MODEL_SAVE_PATH)


── Test results (run 8 pipeline + channel attention) ──
63/63 ━━━━━━━━━━━━━━━━━━━━ 24s 386ms/step - accuracy: 0.8643 - auc: 0.9354 - loss: 0.3964 - precision: 0.8643 - sensitivity: 0.8643 - specificity: 0.8831
  loss: 0.3964
  compile_metrics: 0.8643
